### Importación de datos



In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_1%20.csv"
url2 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_2.csv"
url3 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_3.csv"
url4 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_4.csv"

tienda = pd.read_csv(url)
tienda2 = pd.read_csv(url2)
tienda3 = pd.read_csv(url3)
tienda4 = pd.read_csv(url4)

tiendas = {
    "Tienda 1": tienda,
    "Tienda 2": tienda2,
    "Tienda 3": tienda3,
    "Tienda 4": tienda4
}

tienda.head()



,Producto,Categoría del Producto,Precio,Costo de envío,Fecha de Compra,Vendedor,Lugar de Compra,Calificación,Método de pago,Cantidad de cuotas,lat,lon
0,Asistente virtual,Electrónicos,164300.0,6900.0,16/01/2021,Pedro Gomez,Bogotá,4,Tarjeta de crédito,8,4.60971,-74.08175
1,Mesa de comedor,Muebles,192300.0,8400.0,18/05/2022,Beatriz Morales,Medellín,1,Tarjeta de crédito,4,6.25184,-75.56359
2,Juego de mesa,Juguetes,209600.0,15900.0,15/03/2021,Juan Fernandez,Cartagena,1,Tarjeta de crédito,1,10.39972,-75.51444
3,Microondas,Electrodomésticos,757500.0,41000.0,03/05/2022,Juan Fernandez,Cali,4,Nequi,1,3.43722,-76.52250
4,Silla de oficina,Muebles,335200.0,20200.0,07/11/2020,Maria Alfonso,Medellín,5,Nequi,1,6.25184,-75.56359


#1. Análisis de facturación



In [13]:
facturacion_por_tienda = pd.Series(
    {nombre: df["Precio"].sum() for nombre, df in tiendas.items()},
    name="Facturación total"
).sort_values(ascending=False)

print("Facturación total por tienda:")
print(facturacion_por_tienda.map(lambda x: f"${x:,.2f}"))

Facturación total por tienda:
Tienda 1    $1,150,880,400.00
Tienda 2    $1,116,343,500.00
Tienda 3    $1,098,019,600.00
Tienda 4    $1,038,375,700.00
Name: Facturación total, dtype: object


# 2. Ventas por categoría

In [14]:
ventas_categoria_facturacion = pd.concat(
    [
        df.groupby("Categoría del Producto")["Precio"].sum().rename(nombre)
        for nombre, df in tiendas.items()
    ],
    axis=1
).fillna(0)

print("Facturación por categoría y por tienda:")
print(ventas_categoria_facturacion.applymap(lambda x: f"${x:,.2f}"))

Facturación por categoría y por tienda:
                                Tienda 1         Tienda 2         Tienda 3  \
Categoría del Producto                                                       
Artículos para el hogar   $12,698,400.00   $14,746,900.00   $15,060,000.00   
Deportes y diversión      $39,290,000.00   $34,744,500.00   $35,593,100.00   
Electrodomésticos        $363,685,200.00  $348,567,800.00  $329,237,900.00   
Electrónicos             $429,493,500.00  $410,831,100.00  $410,775,800.00   
Instrumentos musicales    $91,299,000.00  $104,990,300.00   $77,380,900.00   
Juguetes                  $17,995,700.00   $15,945,400.00   $19,401,100.00   
Libros                     $8,784,900.00   $10,091,200.00    $9,498,700.00   
Muebles                  $187,633,700.00  $176,426,300.00  $201,072,100.00   

                                Tienda 4  
Categoría del Producto                    
Artículos para el hogar   $15,074,500.00  
Deportes y diversión      $33,350,100.00  
Electro

/tmp/ipykernel_372/1779962609.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(ventas_categoria_facturacion.applymap(lambda x: f"${x:,.2f}"))


In [15]:
ventas_categoria_cantidad = pd.concat(
    [
        df["Categoría del Producto"].value_counts().rename(nombre)
        for nombre, df in tiendas.items()
    ],
    axis=1
).fillna(0).astype(int)

print("Cantidad de ventas por categoría y por tienda:")
print(ventas_categoria_cantidad)

Cantidad de ventas por categoría y por tienda:
                         Tienda 1  Tienda 2  Tienda 3  Tienda 4
Categoría del Producto                                         
Muebles                       465       442       499       480
Electrónicos                  448       422       451       451
Juguetes                      324       313       315       338
Electrodomésticos             312       305       278       254
Deportes y diversión          284       275       277       277
Instrumentos musicales        182       224       177       170
Libros                        173       197       185       187
Artículos para el hogar       171       181       177       201


# 3. Calificación promedio de la tienda


In [16]:
calificacion_promedio = pd.Series(
    {nombre: df["Calificación"].mean() for nombre, df in tiendas.items()},
    name="Calificación promedio"
).sort_values(ascending=False)

print("Calificación promedio por tienda:")
print(calificacion_promedio.map(lambda x: f"{x:.2f}"))

Calificación promedio por tienda:
Tienda 3    4.05
Tienda 2    4.04
Tienda 4    4.00
Tienda 1    3.98
Name: Calificación promedio, dtype: object


# 4. Productos más y menos vendidos

In [17]:
resumen_productos = []

for nombre, df in tiendas.items():
    conteo_productos = df["Producto"].value_counts()

    max_ventas = int(conteo_productos.max())
    min_ventas = int(conteo_productos.min())

    productos_mas_vendidos = conteo_productos[conteo_productos == max_ventas].index.tolist()
    productos_menos_vendidos = conteo_productos[conteo_productos == min_ventas].index.tolist()

    resumen_productos.append({
        "Tienda": nombre,
        "Producto(s) más vendido(s)": ", ".join(productos_mas_vendidos),
        "Cantidad más vendida": max_ventas,
        "Producto(s) menos vendido(s)": ", ".join(productos_menos_vendidos),
        "Cantidad menos vendida": min_ventas
    })

productos_extremos_df = pd.DataFrame(resumen_productos)

print("Productos más y menos vendidos por tienda:")
print(productos_extremos_df)

Productos más y menos vendidos por tienda:
     Tienda          Producto(s) más vendido(s)  Cantidad más vendida  \
0  Tienda 1  Microondas, TV LED UHD 4K, Armario                    60   
1  Tienda 2           Iniciando en programación                    65   
2  Tienda 3                       Kit de bancas                    57   
3  Tienda 4                            Cama box                    62   

              Producto(s) menos vendido(s)  Cantidad menos vendida  
0  Auriculares con micrófono, Celular ABXY                      33  
1                            Juego de mesa                      32  
2                  Bloques de construcción                      35  
3                       Guitarra eléctrica                      33  


# 5. Envío promedio por tienda

In [18]:
envio_promedio = pd.Series(
    {nombre: df["Costo de envío"].mean() for nombre, df in tiendas.items()},
    name="Envío promedio"
).sort_values()

print("Costo de envío promedio por tienda:")
print(envio_promedio.map(lambda x: f"${x:,.2f}"))

Costo de envío promedio por tienda:
Tienda 4    $23,459.46
Tienda 3    $24,805.68
Tienda 2    $25,216.24
Tienda 1    $26,018.61
Name: Envío promedio, dtype: object


# 6. Comparación y resultado

In [25]:
comparativo_final = pd.DataFrame({
    "Facturación total": {nombre: df["Precio"].sum() for nombre, df in tiendas.items()},
    "Calificación promedio": {nombre: df["Calificación"].mean() for nombre, df in tiendas.items()},
    "Envío promedio": {nombre: df["Costo de envío"].mean() for nombre, df in tiendas.items()}
}).sort_values(by="Facturación total", ascending=False)

print("Comparativo final de tiendas:")
display(
    comparativo_final.style.format({
        "Facturación total": "${:,.2f}",
        "Calificación promedio": "{:.2f}",
        "Envío promedio": "${:,.2f}"
    })
)

Comparativo final de tiendas:


,Facturación total,Calificación promedio,Envío promedio
Tienda 1,"$1,150,880,400.00",3.98,"$26,018.61"
Tienda 2,"$1,116,343,500.00",4.04,"$25,216.24"
Tienda 3,"$1,098,019,600.00",4.05,"$24,805.68"
Tienda 4,"$1,038,375,700.00",4.00,"$23,459.46"


In [27]:
tienda_menor_facturacion = comparativo_final["Facturación total"].idxmin()

print(f"La tienda con menor facturación es: {tienda_menor_facturacion}")

La tienda con menor facturación es: Tienda 4
